# 1. Probability Basics — Reading the System's Output Correctly

**Building a Heart Disease Risk-Screening System — Notebook 1 of 12, Stage 1: Understanding the Raw Signals**

Every notebook in this module works toward one system: a clinic's risk-screening
tool that takes a patient's measurements and outputs *the probability they have
heart disease*. Before touching a single measurement, we need to get comfortable
with what that output — a probability — actually means and how it behaves. Get this
wrong, and every downstream stage of the system (correlation, testing, the model
itself) inherits the confusion.

## The topic

**Probability** quantifies uncertainty on a 0-to-1 scale. In this system, almost
everything eventually reduces to a probability statement: "what fraction of
patients like this one have the disease," "given a positive screening result, how
likely is disease *really*," "if we screen 20 more patients, how many will we
expect to flag."

## Why it matters for this system

A risk score that clinicians and patients don't correctly interpret is worse than
no risk score — it either causes needless alarm or false reassurance. The single
most common way this goes wrong (and the reason this notebook leads with it) is
confusing $P(\text{disease} \mid \text{positive test})$ with $P(\text{positive
test} \mid \text{disease})$ — two different numbers that get swapped constantly in
practice, with real consequences.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(f"{len(df)} patients in the registry")
df[["age", "sex", "chol", "trestbps", "target"]].head()

## The toolkit — what tools are available

| Tool | Answers |
|---|---|
| **Empirical probability** | "What fraction of patients have X?" — just count and divide |
| **Conditional probability** | "What fraction of patients WITH trait B also have X?" |
| **Independence check** | "Does knowing B change anything about X, or not?" |
| **Bayes' theorem** | "I know P(B\|A). What's P(A\|B)?" — flips a conditional around |
| **Law of total probability** | "I know rates within groups. What's the overall rate?" |

## How to choose

The question's *shape* tells you which tool applies: asking about one variable
alone → empirical probability. Asking "…given that…" → conditional probability, in
whichever direction the question is actually phrased. Needing to flip the direction
of a "given that" → Bayes' theorem specifically. Combining known group-level rates
into one number → law of total probability. Getting this matching right is most of
the work; the arithmetic is the easy part.

## Applied to the registry

### Empirical probability: the system's baseline rate

In [ ]:
p_disease = df["target"].mean()
print(f"P(disease) = {p_disease:.3f}  -- the baseline rate before we know anything about a specific patient")

### Conditional probability: does a specific measurement move the needle?

In [ ]:
high_chol = df["chol"] > 240  # clinical threshold for "high" cholesterol

p_given_high_chol = df.loc[high_chol, "target"].mean()
p_given_normal_chol = df.loc[~high_chol, "target"].mean()

print(f"P(disease | high cholesterol)   = {p_given_high_chol:.3f}")
print(f"P(disease | normal cholesterol) = {p_given_normal_chol:.3f}")
print(f"P(disease) overall               = {p_disease:.3f}")

Conditioning on cholesterol moved the probability away from the baseline — that
movement is exactly what makes cholesterol a *useful input* to the eventual system.
A measurement that doesn't move this number at all (checked next) carries no
signal for the system to use.

In [ ]:
p_given_fbs = df.loc[df["fbs"] == 1, "target"].mean()
p_given_no_fbs = df.loc[df["fbs"] == 0, "target"].mean()

print(f"P(disease | high fasting blood sugar)   = {p_given_fbs:.3f}")
print(f"P(disease | normal fasting blood sugar) = {p_given_no_fbs:.3f}")
print(f"\nGap for fasting blood sugar: {abs(p_given_fbs - p_given_no_fbs):.3f}")
print(f"Gap for cholesterol:         {abs(p_given_high_chol - p_given_normal_chol):.3f}")
print("Fasting blood sugar barely moves the needle here -- a real, data-driven preview")
print("of why some inputs end up mattering more to the final system than others.")

### Bayes' theorem: the direction that actually matters for screening

A screening test's **sensitivity** ($P(\text{positive} \mid \text{disease})$) and
**specificity** ($P(\text{negative} \mid \text{no disease})$) are properties of the
*test*, established in validation studies. What a patient and clinician actually
need is the reverse direction: $P(\text{disease} \mid \text{positive})$ — and
Bayes' theorem is the only tool that flips one into the other correctly.

In [ ]:
def positive_predictive_value(prevalence, sensitivity, specificity):
    '''P(disease | positive test), via Bayes' theorem.'''
    p_positive = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    return (sensitivity * prevalence) / p_positive


# A screening test advertised as "90% accurate" both ways, applied at our registry's actual prevalence
registry_prevalence = df["target"].mean()
ppv = positive_predictive_value(prevalence=registry_prevalence, sensitivity=0.90, specificity=0.90)

print(f"Registry disease prevalence: {registry_prevalence:.1%}")
print(f"A 90%-sensitive, 90%-specific screen here has PPV = {ppv:.1%}")
print("Most people's gut-check answer to 'how reliable is a positive result' is close to 90%.")
print("The real answer depends heavily on prevalence -- which is why the SAME test performs")
print("very differently in a general population (low prevalence) than in this clinical")
print("registry (higher prevalence, since it's an enriched sample of people already being")
print("evaluated for heart problems).")

In [ ]:
# Compare against a much rarer general-population prevalence
general_population_ppv = positive_predictive_value(prevalence=0.01, sensitivity=0.90, specificity=0.90)
print(f"Same test at 1% general-population prevalence: PPV = {general_population_ppv:.1%}")
print(f"Same test at this registry's {registry_prevalence:.1%} prevalence: PPV = {ppv:.1%}")
print("\nThis is a systems point, not just a math one: our eventual model's reliability will")
print("depend on WHERE it's deployed, not just how it was built -- a model validated on this")
print("registry could behave very differently if pointed at a general primary-care population.")

### Law of total probability: reconstructing the overall rate from parts

Useful whenever the system needs to combine group-level rates into one number —
e.g. combining age-bracket-specific risk into an overall clinic risk estimate.

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[0, 45, 55, 65, 100], labels=["under 45", "45-55", "55-65", "65+"])
group_rates = df.groupby("age_group", observed=True)["target"].agg(["mean", "count"])
group_rates.columns = ["disease_rate", "n_patients"]
print(group_rates)

weights = group_rates["n_patients"] / group_rates["n_patients"].sum()
reconstructed = (group_rates["disease_rate"] * weights).sum()
print(f"\nReconstructed overall rate from the age groups: {reconstructed:.3f}")
print(f"Actual overall rate:                              {df['target'].mean():.3f}")

## Systems view — what this stage hands to the next one

This notebook established the *output format* the whole system will eventually
speak in: a probability, correctly interpreted, aware of prevalence and direction.
Notebook 2 turns attention to the *inputs* — every measurement in this registry
(age, cholesterol, chest pain type…) is itself a random variable, and understanding
how each one individually behaves is the next required piece before we can combine
them into anything.

## Try it yourself

1. Find a column besides `chol` where conditioning shifts `P(disease)` even more
   than cholesterol did — `thalach` (max heart rate) and `exang` (exercise-induced
   angina) are good candidates.
2. Recompute the PPV for a *better* screening test (95% sensitivity, 95%
   specificity) at this registry's prevalence. How much does a better test help
   compared to simply knowing the right prevalence to apply it to?
3. Reconstruct the overall disease rate from `sex` groups instead of age groups,
   using the law of total probability pattern above.